In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import pyarrow as pa
import pyarrow.parquet as pq
import re
import os
from difflib import get_close_matches

In [2]:
# Importar RIPS
os.listdir('/kaggle/input/datasets/geraldinelaverde')
ruta_archivo = '/kaggle/input/datasets/geraldinelaverde/rips-inicial/RIPS.parquet'
df_rips = pd.read_parquet(ruta_archivo)
df_rips.head()

,Departamento,Municipio,Año,TipoAtencion,Diagnostico,NumeroAtenciones
0,05 - Antioquia,05042 - SantafÃ© De Antioquia,2009.0,CONSULTAS,"A150 - TUBERCULOSIS DEL PULMON, CONFIRMADA POR...",1.0
1,05 - Antioquia,05042 - SantafÃ© De Antioquia,2009.0,CONSULTAS,A159 - TUBERCULOSIS RESPIRATORIA NO ESPECIFICA...,6.0
2,05 - Antioquia,05042 - SantafÃ© De Antioquia,2009.0,CONSULTAS,A418 - OTRAS SEPTICEMIAS ESPECIFICADAS,2.0
3,05 - Antioquia,05042 - SantafÃ© De Antioquia,2009.0,CONSULTAS,"B009 - INFECCION DEBIDA A EL VIRUS DEL HERPES,...",12.0
4,05 - Antioquia,05042 - SantafÃ© De Antioquia,2009.0,CONSULTAS,B182 - HEPATITIS VIRAL TIPO C CRONICA,2.0


In [3]:
#Análisis Exploratorio

print("🔹 SHAPE")
print(df_rips.shape)

print("\n🔹 HEAD")
print(df_rips.head())

print("\n🔹 INFO")
print(df_rips.info())

print("\n🔹 NULOS POR COLUMNA")
print(df_rips.isna().sum().sort_values(ascending=False))

print("\n🔹 TIPOS DE DATOS")
print(df_rips.dtypes)

print("\n🔹 DESCRIPTIVOS NumeroAtenciones")
print(df_rips['NumeroAtenciones'].describe())

# ------------------------------
# Valores únicos
# ------------------------------
print("\n🔹 VALORES ÚNICOS")
for col in ['Departamento','Municipio','TipoAtencion','Diagnostico','Año']:
    print(f'{col}:', df_rips[col].nunique())

# ------------------------------
# Top diagnósticos
# ------------------------------
print("\n🔹 TOP 10 DIAGNÓSTICOS")
print(
    df_rips.groupby('Diagnostico')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
           .head(10)
)

# ------------------------------
# Atenciones por departamento
# ------------------------------
print("\n🔹 ATENCIONES POR DEPARTAMENTO")
print(
    df_rips.groupby('Departamento')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
)

# ------------------------------
# Tipo de atención
# ------------------------------
print("\n🔹 ATENCIONES POR TIPO")
print(
    df_rips.groupby('TipoAtencion')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
)

# ------------------------------
# Municipios con más carga
# ------------------------------
print("\n🔹 TOP 15 MUNICIPIOS")
print(
    df_rips.groupby('Municipio')['NumeroAtenciones']
           .sum()
           .sort_values(ascending=False)
           .head(15)
)

🔹 SHAPE
(24322985, 6)

🔹 HEAD
     Departamento                      Municipio     Año TipoAtencion  \
0  05 - Antioquia  05042 - SantafÃ© De Antioquia  2009.0    CONSULTAS   
1  05 - Antioquia  05042 - SantafÃ© De Antioquia  2009.0    CONSULTAS   
2  05 - Antioquia  05042 - SantafÃ© De Antioquia  2009.0    CONSULTAS   
3  05 - Antioquia  05042 - SantafÃ© De Antioquia  2009.0    CONSULTAS   
4  05 - Antioquia  05042 - SantafÃ© De Antioquia  2009.0    CONSULTAS   

                                         Diagnostico  NumeroAtenciones  
0  A150 - TUBERCULOSIS DEL PULMON, CONFIRMADA POR...               1.0  
1  A159 - TUBERCULOSIS RESPIRATORIA NO ESPECIFICA...               6.0  
2             A418 - OTRAS SEPTICEMIAS ESPECIFICADAS               2.0  
3  B009 - INFECCION DEBIDA A EL VIRUS DEL HERPES,...              12.0  
4              B182 - HEPATITIS VIRAL TIPO C CRONICA               2.0  

🔹 INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24322985 entries, 0 to 24322984
Dat

In [4]:
#Limpieza Inicial

# =========================================
# LIMPIEZA ESTRUCTURAL df_rips
# =========================================

import pandas as pd

# 1) Corregir codificación de textos
def fix_encoding(text):
    try:
        return text.encode('latin1').decode('utf-8')
    except:
        return text

for col in ['Departamento','Municipio','TipoAtencion','Diagnostico']:
    df_rips[col] = df_rips[col].astype(str).apply(fix_encoding)

# 2) Quitar basura que se metió como filas
df_rips = df_rips[~df_rips['Departamento'].str.contains('message|error|status|{|}', regex=True)]

# 3) Corregir tipos de datos
df_rips['Año'] = df_rips['Año'].astype('Int64')
df_rips['NumeroAtenciones'] = df_rips['NumeroAtenciones'].astype('Int64')

# 4) Eliminar nulos reales
df_rips = df_rips.dropna()

# 5) Reset index
df_rips = df_rips.reset_index(drop=True)

print("✅ Limpieza estructural terminada")
print(df_rips.info())
print(df_rips.head())

✅ Limpieza estructural terminada
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24322980 entries, 0 to 24322979
Data columns (total 6 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   Departamento      object
 1   Municipio         object
 2   Año               Int64 
 3   TipoAtencion      object
 4   Diagnostico       object
 5   NumeroAtenciones  Int64 
dtypes: Int64(2), object(4)
memory usage: 1.1+ GB
None
     Departamento                     Municipio   Año TipoAtencion  \
0  05 - Antioquia  05042 - Santafé De Antioquia  2009    CONSULTAS   
1  05 - Antioquia  05042 - Santafé De Antioquia  2009    CONSULTAS   
2  05 - Antioquia  05042 - Santafé De Antioquia  2009    CONSULTAS   
3  05 - Antioquia  05042 - Santafé De Antioquia  2009    CONSULTAS   
4  05 - Antioquia  05042 - Santafé De Antioquia  2009    CONSULTAS   

                                         Diagnostico  NumeroAtenciones  
0  A150 - TUBERCULOSIS DEL PULMON, CONFIRMADA POR...            

In [5]:
#Importar divipola
ruta_davipola = '/kaggle/input/datasets/nicolasacostaa/municipios-divipola/DIVIPOLA_Municipios.xlsx - Municipios.csv'

df_Davipola = pd.read_csv(ruta_davipola, encoding='utf-8')

df_Davipola = df_Davipola.rename(columns={
    'doc_dep':      'CÓDIGO DEPARTAMENTO',
    'departamento': 'NOMBRE DEPARTAMENTO',
    'cod_muni':     'CÓDIGO MUNICIPIO',
    'municipio':    'NOMBRE MUNICIPIO',
})

df_Davipola['CÓDIGO DEPARTAMENTO'] = df_Davipola['CÓDIGO DEPARTAMENTO'].astype(str).str.zfill(2)
df_Davipola['CÓDIGO MUNICIPIO']    = df_Davipola['CÓDIGO MUNICIPIO'].astype(str).str.zfill(5)



In [6]:
# =========================================
# RIPS + DIVIPOLA → DATASET GEOGRAFICO LIMPIO
# =========================================
# -------------------------------------------------
# 1) Extraer códigos reales desde RIPS (NO textos)
# -------------------------------------------------

df_rips['CÓDIGO DEPARTAMENTO'] = df_rips['Departamento'].str[:2]
df_rips['CÓDIGO MUNICIPIO']    = df_rips['Municipio'].str[:5]

df_rips[['Cod_Diagnostico','Diagnostico_Nombre']] = \
    df_rips['Diagnostico'].str.split(' - ', n=1, expand=True)



In [7]:
# -------------------------------------------------
# 3) Unir con DIVIPOLA (la parte clave)
# -------------------------------------------------

df_rips = df_rips.merge(
    df_Davipola,
    on=['CÓDIGO DEPARTAMENTO','CÓDIGO MUNICIPIO'],
    how='left'
)

In [8]:
# -------------------------------------------------
# 4) Crear ANO y MES desde Año (para tu modelo)
# -------------------------------------------------

df_rips['ANO'] = df_rips['Año'].astype(int)
df_rips['MES'] = 1  # RIPS no trae mes, se deja fijo si luego lo cruzas con giros
# Guardar dataframe como parquet
df_rips.to_parquet('/kaggle/working/df_rips_1.parquet', index=False)